In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
import ROOT
from analysis_framework import Dataset
from ObjectSelectionHelper import ObjectSelectionHelper, make_lvec_M, make_lvec_E

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8d5cd60


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# prod = False
prod = True
no_rvec = True
# write_outputs = False
write_outputs = True
# plot_dir_postfix = "-new-cuts"
dataset_path = "data/datasets/pre-selected/checked-test.json"
output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/selected-objects/test"
output_meta_path = "data/datasets/selected-objects"
output_meta = f"{output_meta_path}/test.json"
checked_output_meta = f"{output_meta_path}/checked-test.json"
# output_collections = [
#     "true_lep_lvec", "true_nu_lvec", "true_quark1_lvec", "true_quark2_lvec",
#     "iso_lep_lvec", "nu_lvec", "R2Jet_sel1_lvec", "R2Jet_sel2_lvec",
#     ]
# true lvecs do not exist in every df so cannot be explicitly requested...
# urgh but empty snapshots are also not allowed
output_collections = r"(true_\w+_lvec)|(true_lep_charge)|(true_beam_\w+_lvec)|(iso_lep_lvec)|(iso_lep_charge)|(nu_lvec)|(R2Jet_sel1_lvec)|(R2Jet_sel2_lvec)|(R2Jet1_lvec)|(R2Jet2_lvec)"
# plot_dir = f"plots/pre-selection/test{plot_dir_postfix}"
if prod:
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/processed-no-exc-higgs-min-aa-min-higgs.json"
    # dataset_path = "data/datasets/miniDSTs/min-higgs.json"
    dataset_path = "data/datasets/pre-selected/checked-signal-only.json"
    # output_path = "root://eospublic.cern.ch//eos/experiment/clicdp/data/user/l/lreichen/snapshots3/min-higgs-d"
    output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/selected-objects/signal-only"
    output_meta_path = "data/datasets/selected-objects"
    output_meta = f"{output_meta_path}/signal-only.json"
    checked_output_meta = f"{output_meta_path}/checked-signal-only.json"
    # plot_dir = "plots/pre-selection/full"
    # plot_dir = f"plots/pre-selection/min-higgs{plot_dir_postfix}"


In [4]:
ROOT.EnableImplicitMT(n_threads)

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ObjectSelectionHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xc37c150


In [7]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [8]:
ROOT.gStyle.SetOptStat(1)

In [9]:
# import json
# print(json.dumps(analysis._categories, indent=2))

In [10]:
analysis.define_truth_objects(signal_category)

In [11]:
analysis.define_only_on(signal_category, "true_nu_Pz", "true_nu_lvec.Pz()")

In [12]:
# make nominal iso lep
# try to make iso lep + brems, try to make cheated brems
analysis.Define("iso_lep_idx", "IsolatedElectrons_objIdx.index[0]")
analysis.Define("iso_lep_charge", "PandoraPFOs.charge[iso_lep_idx]")
analysis.Define("iso_lep_lvec", "ROOT::Math::PxPyPzEVector(PandoraPFOs.momentum.x[iso_lep_idx], PandoraPFOs.momentum.y[iso_lep_idx], PandoraPFOs.momentum.z[iso_lep_idx], PandoraPFOs.energy[iso_lep_idx])")
analysis.Define("iso_lep_E", "iso_lep_lvec.energy()")
analysis.Define("iso_lep_M", "iso_lep_lvec.M()")

analysis.Define("n_iso_gammas", "IsolatedPhotons_objIdx.index.size()")
analysis.Define("iso_gamma_idx", "IsolatedPhotons_objIdx.index[0]")
analysis.Define("iso_gamma_lvec", "ROOT::Math::PxPyPzEVector(PandoraPFOs.momentum.x[iso_gamma_idx], PandoraPFOs.momentum.y[iso_gamma_idx], PandoraPFOs.momentum.z[iso_gamma_idx], PandoraPFOs.energy[iso_gamma_idx])")

# analysis.Define("iso_lep_gamma_deltaR", "ROOT::Math::VectorUtil::DeltaR(iso_lep_lvec, iso_gamma_lvec)")
analysis.Define("iso_lep_gamma_angle", "ROOT::Math::VectorUtil::Angle(iso_lep_lvec, iso_gamma_lvec)")
analysis.Define("iso_lep_gamma_close", "iso_lep_gamma_angle > 0. && iso_lep_gamma_angle < 0.01")

analysis.Define("iso_lep_gamma_lvec", "n_iso_gammas > 0 && iso_lep_gamma_close ? iso_lep_lvec + iso_gamma_lvec : iso_lep_lvec")

In [13]:
# brems recovery a la Sang Hyun
analysis.Define("PFO_lvecs", "Construct<ROOT::Math::PxPyPzEVector>(PandoraPFOs.momentum.x, PandoraPFOs.momentum.y, PandoraPFOs.momentum.z, PandoraPFOs.energy)")
analysis.Define("PFO_etas", "return Map(PFO_lvecs, [] (const auto &el) {return el.eta();} )")
analysis.Define("PFO_phis", "return Map(PFO_lvecs, [] (const auto &el) {return el.phi();} )")
analysis.Define("brems_PFOs", "PandoraPFOs.PDG == 22 && abs(PFO_etas - iso_lep_lvec.eta()) <= 0.005 &&  abs(PFO_phis - iso_lep_lvec.phi()) <= 0.05")

analysis.Define("iso_lep_brems_lvec", "iso_lep_lvec + Sum(PFO_lvecs[brems_PFOs], ROOT::Math::PxPyPzEVector())")
# TODO: do something with this!
# I.e. remove identified brems from jets
# might also be worth to consider to add other isolated particles back to jets somehow...

In [14]:
analysis.define_only_on(signal_category, "true_lep_E", "true_lep_lvec.energy()")

analysis.define_deltas("iso_lep", "iso_lep_lvec", "true_lep_lvec", signal_category)
analysis.define_deltas("iso_lep_gamma", "iso_lep_gamma_lvec", "true_lep_lvec", signal_category)
analysis.define_deltas("iso_lep_brems", "iso_lep_brems_lvec", "true_lep_lvec", signal_category)

analysis.define_only_on(signal_category, "iso_lep_delta_phi_q", "iso_lep_delta_phi * true_lep_charge")
analysis.define_only_on(signal_category, "iso_lep_gamma_delta_phi_q", "iso_lep_gamma_delta_phi * true_lep_charge")
analysis.define_only_on(signal_category, "iso_lep_brems_delta_phi_q", "iso_lep_brems_delta_phi * true_lep_charge")


In [15]:
ROOT.gInterpreter.Declare("#include \"analyzers.h\"")
# make nominal jets, clean jets, try to make cheated super clean jets
analysis.Define("R2Jet_lvecs", "return Map(Refined2Jets, [] (const auto& el) {return ROOT::Math::PxPyPzEVector(el.momentum.x, el.momentum.y, el.momentum.z, el.energy);})")
analysis.Define("clean_R2Jets", "remove_constituents(Refined2Jets, _Refined2Jets_particles, PandoraPFOs, PFO_ovl_idx)")
analysis.Define("clean_R2Jet_lvecs", "return Map(clean_R2Jets, [] (const auto& el) {return ROOT::Math::PxPyPzEVector(el.momentum.x, el.momentum.y, el.momentum.z, el.energy);})")

ROOT.gInterpreter.Declare("#include <edm4hep/MCParticle.h>")
ROOT.gInterpreter.Declare("#include <edm4hep/utils/bit_utils.h>")
analysis.Define("mcp_ovl_mask", "return Map(MCParticlesSkimmed.simulatorStatus, [] (const auto simStat) { return edm4hep::utils::checkBit(simStat, edm4hep::MCParticle::BITOverlay);})")
analysis.Define("pfo_ovl_mask", "mcp_mask_to_pfo_mask(mcp_ovl_mask, PandoraPFOs, _RecoMCTruthLink_from, _RecoMCTruthLink_to, RecoMCTruthLink_weight)")

analysis.Define("cheat_clean_R2Jets", "remove_constituents(Refined2Jets, _Refined2Jets_particles, PandoraPFOs, pfo_ovl_mask)")
analysis.Define("cheat_clean_R2Jet_lvecs", "return Map(cheat_clean_R2Jets, [] (const auto& el) {return ROOT::Math::PxPyPzEVector(el.momentum.x, el.momentum.y, el.momentum.z, el.energy);})")

analysis.Define("R2Jet1_lvec", "R2Jet_lvecs[0]")
analysis.Define("R2Jet2_lvec", "R2Jet_lvecs[1]")
analysis.Define("clean_R2Jet1_lvec", "clean_R2Jet_lvecs[0]")
analysis.Define("clean_R2Jet2_lvec", "clean_R2Jet_lvecs[1]")
analysis.Define("cheat_clean_R2Jet1_lvec", "cheat_clean_R2Jet_lvecs[0]")
analysis.Define("cheat_clean_R2Jet2_lvec", "cheat_clean_R2Jet_lvecs[1]")

# also define 3d angular distance between jet and q
analysis.define_only_on(signal_category, "R2Jet1_q1_angle", "ROOT::Math::VectorUtil::Angle(R2Jet1_lvec, true_quark1_lvec)")
analysis.define_only_on(signal_category, "R2Jet1_q2_angle", "ROOT::Math::VectorUtil::Angle(R2Jet1_lvec, true_quark2_lvec)")
analysis.define_only_on(signal_category, "R2Jet2_q1_angle", "ROOT::Math::VectorUtil::Angle(R2Jet2_lvec, true_quark1_lvec)")
analysis.define_only_on(signal_category, "R2Jet2_q2_angle", "ROOT::Math::VectorUtil::Angle(R2Jet2_lvec, true_quark2_lvec)")

# assign as correct the combination with smallest total angular distance
analysis.define_only_on(signal_category, "R2Jet_cmb1_scr", "R2Jet1_q1_angle * R2Jet2_q2_angle")
analysis.define_only_on(signal_category, "R2Jet_cmb2_scr", "R2Jet1_q2_angle * R2Jet2_q1_angle")

# smaller score == better
analysis.define_only_on(signal_category, "R2Jet_sel1_lvec", "R2Jet_cmb1_scr < R2Jet_cmb2_scr ? R2Jet1_lvec : R2Jet2_lvec")
analysis.define_only_on(signal_category, "R2Jet_sel2_lvec", "R2Jet_cmb1_scr < R2Jet_cmb2_scr ? R2Jet2_lvec : R2Jet1_lvec")

# use the same combination for the clean jets
analysis.define_only_on(signal_category, "clean_R2Jet_sel1_lvec", "R2Jet_cmb1_scr < R2Jet_cmb2_scr ? clean_R2Jet1_lvec : clean_R2Jet2_lvec")
analysis.define_only_on(signal_category, "clean_R2Jet_sel2_lvec", "R2Jet_cmb1_scr < R2Jet_cmb2_scr ? clean_R2Jet2_lvec : clean_R2Jet1_lvec")
analysis.define_only_on(signal_category, "cheat_clean_R2Jet_sel1_lvec", "R2Jet_cmb1_scr < R2Jet_cmb2_scr ? cheat_clean_R2Jet1_lvec : cheat_clean_R2Jet2_lvec")
analysis.define_only_on(signal_category, "cheat_clean_R2Jet_sel2_lvec", "R2Jet_cmb1_scr < R2Jet_cmb2_scr ? cheat_clean_R2Jet2_lvec : cheat_clean_R2Jet1_lvec")

# define deltas for better combination
analysis.define_deltas("R2Jet_sel1", "R2Jet_sel1_lvec", "true_quark1_lvec", signal_category)
analysis.define_deltas("R2Jet_sel2", "R2Jet_sel2_lvec", "true_quark2_lvec", signal_category)

analysis.define_only_on(signal_category, "R2Jet_sel1_Pt", "R2Jet_sel1_lvec.Pt()")
analysis.define_only_on(signal_category, "R2Jet_sel1_Px", "R2Jet_sel1_lvec.Px()")
analysis.define_only_on(signal_category, "R2Jet_sel1_Py", "R2Jet_sel1_lvec.Py()")
analysis.define_only_on(signal_category, "R2Jet_sel2_Pt", "R2Jet_sel2_lvec.Pt()")
analysis.define_only_on(signal_category, "R2Jet_sel2_Px", "R2Jet_sel2_lvec.Px()")
analysis.define_only_on(signal_category, "R2Jet_sel2_Py", "R2Jet_sel2_lvec.Py()")

analysis.define_only_on(signal_category, "true_quark1_Pt", "true_quark1_lvec.Pt()")
analysis.define_only_on(signal_category, "true_quark1_Px", "true_quark1_lvec.Px()")
analysis.define_only_on(signal_category, "true_quark1_Py", "true_quark1_lvec.Py()")
analysis.define_only_on(signal_category, "true_quark2_Pt", "true_quark2_lvec.Pt()")
analysis.define_only_on(signal_category, "true_quark2_Px", "true_quark2_lvec.Px()")
analysis.define_only_on(signal_category, "true_quark2_Py", "true_quark2_lvec.Py()")

analysis.define_deltas("clean_R2Jet_sel1", "clean_R2Jet_sel1_lvec", "true_quark1_lvec", signal_category)
analysis.define_deltas("clean_R2Jet_sel2", "clean_R2Jet_sel2_lvec", "true_quark2_lvec", signal_category)
analysis.define_deltas("cheat_clean_R2Jet_sel1", "cheat_clean_R2Jet_sel1_lvec", "true_quark1_lvec", signal_category)
analysis.define_deltas("cheat_clean_R2Jet_sel2", "cheat_clean_R2Jet_sel2_lvec", "true_quark2_lvec", signal_category)

analysis.define_deltas("true_quark1_postCRC", "true_quark1_postCRC_lvec", "true_quark1_lvec", signal_category)
analysis.define_deltas("true_quark2_postCRC", "true_quark2_postCRC_lvec", "true_quark2_lvec", signal_category)
analysis.define_deltas("true_hadronic_W_postCRC", "true_hadronic_W_postCRC_lvec", "true_hadronic_W_lvec", signal_category)

In [16]:
analysis.Define("R2Jet1_M", "R2Jet1_lvec.M()")
analysis.Define("R2Jet2_M", "R2Jet2_lvec.M()")

In [17]:
for name in ["", "clean_", "cheat_clean_"]:
    # TODO: what about other seen energy here?? e.g. from other PFOs classified as isolated or from the BCal?
    # would need to also merge that into one of the other objects down the road to keep event consistent for the matrix element...
    analysis.Define(f"{name}visible_lvec", f"iso_lep_lvec + {name}R2Jet1_lvec + {name}R2Jet2_lvec")
    analysis.Define(f"{name}diff_lvec", f"ROOT::Math::PxPyPzEVector(250*sin({x_angle/2}), 0., 0., 250.) - {name}visible_lvec")
    analysis.Define(f"{name}nu_lvec", f"{name}diff_lvec")

    analysis.define_deltas(f"{name}nu", f"{name}nu_lvec", "true_nu_lvec", signal_category)

    analysis.Define(f"{name}nu_M", f"{name}nu_lvec.M()")
    analysis.Define(f"{name}nu_Pz", f"{name}nu_lvec.Pz()")

analysis.define_only_on(signal_category, "cheat_ISR_clean_nu_lvec", f"ROOT::Math::PxPyPzEVector(250*sin({x_angle/2}), 0., 0., 250.) - iso_lep_lvec - cheat_clean_R2Jet1_lvec - cheat_clean_R2Jet2_lvec - true_isr1_lvec - true_isr2_lvec")
analysis.define_deltas("cheat_ISR_clean_nu", "cheat_ISR_clean_nu_lvec", "true_nu_lvec", signal_category)
analysis.define_only_on(signal_category, "cheat_ISR_clean_nu_M", "cheat_ISR_clean_nu_lvec.M()")

In [18]:
# more ISR stuff
analysis.define_only_on(signal_category, "true_isr_sum_lvec", "true_isr1_lvec + true_isr2_lvec")
analysis.define_only_on(signal_category, "true_isr_sum_M", "true_isr_sum_lvec.M()")
analysis.define_only_on(signal_category, "true_isr_sum_Pt", "true_isr_sum_lvec.Pt()")
analysis.define_only_on(signal_category, "true_isr_sum_Pz", "true_isr_sum_lvec.Pz()")
analysis.define_only_on(signal_category, "true_isr_max_Pz", "abs(true_isr1_lvec.Pz()) > abs(true_isr2_lvec.Pz()) ? true_isr1_lvec.Pz() : true_isr2_lvec.Pz()")
analysis.define_only_on(signal_category, "true_isr_min_E", "std::min(true_isr1_lvec.E(), true_isr2_lvec.E())")

In [19]:
# nu reco taking isr into account
analysis.Define("gamma1_Pz", "diff_lvec.M2() / (2*(-diff_lvec.Pz() - diff_lvec.E()))")
analysis.Define("gamma2_Pz", "diff_lvec.M2() / (2*(-diff_lvec.Pz() + diff_lvec.E()))")

analysis.Define("gamma1_lvec", "ROOT::Math::PxPyPzEVector(0., 0., gamma1_Pz, abs(gamma1_Pz))")
analysis.Define("gamma2_lvec", "ROOT::Math::PxPyPzEVector(0., 0., gamma2_Pz, abs(gamma2_Pz))")

analysis.define_deltas("gamma1", "gamma1_lvec", "true_isr_sum_lvec", signal_category)
analysis.define_deltas("gamma2", "gamma2_lvec", "true_isr_sum_lvec", signal_category)

analysis.define_only_on(signal_category, "gamma_lvec", "abs(gamma1_delta_Pz) < abs(gamma2_delta_Pz) ? gamma1_lvec : gamma2_lvec")
analysis.define_only_on(signal_category, "wrong_gamma_lvec", "abs(gamma1_delta_Pz) > abs(gamma2_delta_Pz) ? gamma1_lvec : gamma2_lvec")

analysis.Define("gamma1_nu_lvec", "diff_lvec - gamma1_lvec")
analysis.Define("gamma2_nu_lvec", "diff_lvec - gamma2_lvec")

analysis.define_only_on(signal_category, "gamma_nu_lvec", "diff_lvec - gamma_lvec")
analysis.define_only_on(signal_category, "wrong_gamma_nu_lvec", "diff_lvec - wrong_gamma_lvec")
analysis.define_deltas("gamma_nu", "gamma_nu_lvec", "true_nu_lvec", signal_category)
analysis.define_deltas("wrong_gamma_nu", "wrong_gamma_nu_lvec", "true_nu_lvec", signal_category)
analysis.define_only_on(signal_category, "gamma_nu_M", "gamma_nu_lvec.M()")
analysis.define_only_on(signal_category, "wrong_gamma_nu_M", "wrong_gamma_nu_lvec.M()")
analysis.define_only_on(signal_category, "gamma_nu_Pz", "gamma_nu_lvec.Pz()")
analysis.define_only_on(signal_category, "wrong_gamma_nu_Pz", "wrong_gamma_nu_lvec.Pz()")

analysis.define_only_on(signal_category, "gamma_nu_lep_lvec", "gamma_nu_lvec + iso_lep_lvec")
analysis.define_only_on(signal_category, "wrong_gamma_nu_lep_lvec", "wrong_gamma_nu_lvec + iso_lep_lvec")

analysis.define_only_on(signal_category, "gamma_nu_lep_M", "gamma_nu_lep_lvec.M()")
analysis.define_only_on(signal_category, "wrong_gamma_nu_lep_M", "wrong_gamma_nu_lep_lvec.M()")

analysis.define_only_on(signal_category, "true_nu_lep_lvec", "true_nu_lvec + true_lep_lvec")
analysis.define_only_on(signal_category, "true_nu_lep_M", "true_nu_lep_lvec.M()")


analysis.define_only_on(signal_category, "gamma_Pz", "gamma_lvec.Pz()")
analysis.define_only_on(signal_category, "wrong_gamma_Pz", "wrong_gamma_lvec.Pz()")

analysis.define_only_on(signal_category, "gamma_PzQ", "gamma_Pz * iso_lep_charge")
analysis.define_only_on(signal_category, "wrong_gamma_PzQ", "wrong_gamma_Pz * iso_lep_charge")

# check diff to missing Pz
analysis.define_only_on(signal_category, "gamma_Pz_diff", "gamma_Pz - diff_lvec.Pz()")
analysis.define_only_on(signal_category, "wrong_gamma_Pz_diff", "wrong_gamma_Pz - diff_lvec.Pz()")
analysis.define_only_on(signal_category, "gamma_PzQ_diff", "gamma_Pz_diff * iso_lep_charge")
analysis.define_only_on(signal_category, "wrong_gamma_PzQ_diff", "wrong_gamma_Pz_diff * iso_lep_charge")

In [20]:
analysis.book_histogram_1D("true_lep_E", "true_lep_E", ("", "true lep E", 125, 0., 125.), categories=signal_category)

for name in ["iso_lep", "iso_lep_gamma", "iso_lep_brems"]:
    analysis.book_histogram_1D(f"{name}_delta_E", f"{name}_delta_E", ("", f";{name} #Delta E [GeV]", 150, -10., 5.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_P", f"{name}_delta_P", ("", f";{name} #Delta P [GeV]", 150, -5., 5.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_theta", f"{name}_delta_theta", ("", f";{name} #Delta #theta [rad]", 150, -0.0005, 0.0005), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_phi", f"{name}_delta_phi", ("", f";{name} #Delta #phi [rad]", 150, -0.0005, 0.0005), categories=signal_category)

analysis.book_histogram_1D("iso_lep_delta_phi_q", "iso_lep_delta_phi_q", ("", ";iso lep #Delta #phi #dot q [rad]", 150, -0.0005, 0.0005), categories=signal_category)
analysis.book_histogram_1D("iso_lep_gamma_delta_phi_q", "iso_lep_gamma_delta_phi_q", ("", ";iso lep #Delta #phi #dot q [rad]", 150, -0.0005, 0.0005), categories=signal_category)
analysis.book_histogram_1D("iso_lep_brems_delta_phi_q", "iso_lep_brems_delta_phi_q", ("", ";iso lep #Delta #phi #dot q [rad]", 150, -0.0005, 0.0005), categories=signal_category)

# analysis.book_histogram_1D("iso_lep_gamma_deltaR", "iso_lep_gamma_deltaR", ("", ";#Delta R lep gamma", 300, 0., 0.02), categories=signal_category)
analysis.book_histogram_1D("iso_lep_gamma_angle", "iso_lep_gamma_angle", ("", ";Angle lep gamma", 300, 0., 0.02), categories=signal_category)

In [21]:
for name in ["R2Jet_sel1", "R2Jet_sel2", "clean_R2Jet_sel1", "clean_R2Jet_sel2", "cheat_clean_R2Jet_sel1", "cheat_clean_R2Jet_sel2", "true_quark1_postCRC", "true_quark2_postCRC", "true_hadronic_W_postCRC"]:
    analysis.book_histogram_1D(f"{name}_delta_E", f"{name}_delta_E", ("", f";{name} #Delta E [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_P", f"{name}_delta_P", ("", f";{name} #Delta P [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Px", f"{name}_delta_Px", ("", f";{name} #Delta Px [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_PxPy", f"{name}_delta_PxPy", ("", f";{name} #Delta Px+Py [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_PxPy2", f"{name}_delta_PxPy2", ("", f";{name} #Delta Px2+Py2 [GeV]", 150, -300., 300.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Pxy", f"{name}_delta_Pxy", ("", f";{name} #Delta Pxy [GeV]", 150, -300., 300.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Py", f"{name}_delta_Py", ("", f";{name} #Delta Py [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Pz", f"{name}_delta_Pz", ("", f";{name} #Delta Pz [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Pt", f"{name}_delta_Pt", ("", f";{name} #Delta Pt [GeV]", 150, -15., 15.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Pt2", f"{name}_delta_Pt2", ("", f";{name} #Delta Pt2 [GeV]", 150, -300., 300.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_theta", f"{name}_delta_theta", ("", f";{name} #Delta #theta [rad]", 300, -0.15, 0.15), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_phi", f"{name}_delta_phi", ("", f";{name} #Delta #phi [rad]", 300, -0.15, 0.15), categories=signal_category)

In [22]:
for name in ["nu", "clean_nu", "cheat_clean_nu", "cheat_ISR_clean_nu", "gamma_nu", "wrong_gamma_nu"]:
    analysis.book_histogram_1D(f"{name}_delta_E", f"{name}_delta_E", ("", ";#nu #Delta E [GeV]", 150, -25., 25.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_theta", f"{name}_delta_theta", ("", ";#nu #Delta #theta [rad]", 150, -0.2, 0.2), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_phi", f"{name}_delta_phi", ("", ";#nu #Delta #phi [rad]", 150, -0.2, 0.2), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_P", f"{name}_delta_P", ("", ";#nu #Delta P [GeV]", 150, -25., 25.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_P_scaled", f"{name}_delta_P_scaled", ("", ";#nu #sigma P / sqrt(P) [GeV]", 150, -.25, .25), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Pt", f"{name}_delta_Pt", ("", ";#nu #Delta Pt [GeV]", 150, -25., 25.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_delta_Pz", f"{name}_delta_Pz", ("", ";#nu #Delta Pz [GeV]", 150, -25., 25.), categories=signal_category)
    analysis.book_histogram_1D(f"{name}_M", f"{name}_M", ("", ";#nu M [GeV]", 150, -50., 50.), categories=signal_category)

In [23]:
analysis.book_histogram_1D("iso_lep_E", "iso_lep_E", ("", "iso lep E", 125, 0., 125.))
analysis.book_histogram_1D("true_isr_sum_M", "true_isr_sum_M", ("", ";ISR sum M [GeV]", 150, 0., 1.), categories=signal_category)
analysis.book_histogram_1D("true_isr_sum_Pt", "true_isr_sum_Pt", ("", ";ISR sum Pt [GeV]", 150, 0., 50.), categories=signal_category)
analysis.book_histogram_1D("true_isr_min_E", "true_isr_min_E", ("", ";ISR min E [GeV]", 150, 0., 1.), categories=signal_category)

analysis.book_histogram_1D("iso_lep_M", "iso_lep_M", ("", "iso lep M", 500, -.05, .15))
analysis.book_histogram_1D("R2Jet1_M", "R2Jet1_M", ("", "R2Jet1 M", 500, 0., 100.))
analysis.book_histogram_1D("R2Jet2_M", "R2Jet2_M", ("", "R2Jet2 M", 500, 0., 100.))

In [24]:
analysis.book_histogram_1D("gamma_nu_lep_M", "gamma_nu_lep_M", ("", ";M e#nu [GeV]", 500, -250., 250.), categories=signal_category)
analysis.book_histogram_1D("wrong_gamma_nu_lep_M", "wrong_gamma_nu_lep_M", ("", ";M e#nu [GeV]", 500, -250., 250.), categories=signal_category)
analysis.book_histogram_1D("true_nu_lep_M", "true_nu_lep_M", ("", ";M e#nu [GeV]", 500, -250., 250.), categories=signal_category)

analysis.book_histogram_1D("true_isr_sum_Pz", "true_isr_sum_Pz", ("", ";ISR sum Pz [GeV]", 250, -125., 125.), categories=signal_category)

analysis.book_histogram_2D("isr_Pz_rvst", "true_isr_sum_Pz", "gamma_Pz", ("", ";true ISR sum Pz [GeV]; reco ISR Pz [GeV]", 125, -125., 125., 125, -125., 125.), categories=signal_category)
analysis.book_histogram_2D("isr_Pz_wrvst", "true_isr_sum_Pz", "wrong_gamma_Pz", ("", ";true ISR sum Pz [GeV]; wrong reco ISR Pz [GeV]", 125, -125., 125., 125, -125., 125.), categories=signal_category)

analysis.book_histogram_1D("gamma_Pz", "gamma_Pz", ("", ";Pz #gamma [GeV]", 250, -125., 125.), categories=signal_category)
analysis.book_histogram_1D("wrong_gamma_Pz", "wrong_gamma_Pz", ("", ";Pz wrong #gamma [GeV]", 250, -125., 125.), categories=signal_category)
analysis.book_histogram_1D("gamma_PzQ", "gamma_PzQ", ("", ";PzQ #gamma [GeV]", 250, -125., 125.), categories=signal_category)
analysis.book_histogram_1D("wrong_gamma_PzQ", "wrong_gamma_PzQ", ("", ";PzQ wrong #gamma [GeV]", 250, -125., 125.), categories=signal_category)

analysis.book_histogram_1D("gamma_Pz_diff", "gamma_Pz_diff", ("", ";diff Pz #gamma [GeV]", 200, -100., 100.), categories=signal_category)
analysis.book_histogram_1D("wrong_gamma_Pz_diff", "wrong_gamma_Pz_diff", ("", ";diff Pz wrong #gamma [GeV]", 200, -100., 100.), categories=signal_category)
analysis.book_histogram_1D("gamma_PzQ_diff", "gamma_PzQ_diff", ("", ";diff PzQ #gamma [GeV]", 200, -100., 100.), categories=signal_category)
analysis.book_histogram_1D("wrong_gamma_PzQ_diff", "wrong_gamma_PzQ_diff", ("", ";diff PzQ wrong #gamma [GeV]", 200, -100., 100.), categories=signal_category)

In [25]:
analysis.book_histogram_2D("nu_Pz_vst", "true_nu_Pz", "nu_Pz", ("", ";true nu Pz [GeV]; reco nu Pz [GeV]", 125, -125., 125., 125, -125., 125.), categories=signal_category)
analysis.book_histogram_2D("gamma_nu_Pz_vst", "true_nu_Pz", "gamma_nu_Pz", ("", ";true nu Pz [GeV]; reco gamma nu Pz [GeV]", 125, -125., 125., 125, -125., 125.), categories=signal_category)

In [26]:
analysis.book_histogram_2D("R2Jet_sel1_delta_Pt_vs_Pt", "R2Jet_sel1_Pt", "R2Jet_sel1_delta_Pt", ("", ";Jet1 reco Pt [GeV]; Jet1 #Delta reco Pt [GeV]", 125, 0., 125., 150, -15., 15.), categories=signal_category)
analysis.book_histogram_2D("R2Jet_sel2_delta_Pt_vs_Pt", "R2Jet_sel2_Pt", "R2Jet_sel2_delta_Pt", ("", ";Jet2 reco Pt [GeV]; Jet2 #Delta reco Pt [GeV]", 125, 0., 125., 150, -15., 15.), categories=signal_category)

analysis.book_histogram_2D("true_quark1_Pt_vs_R2Jet_sel1_Pt", "true_quark1_Pt", "R2Jet_sel1_Pt", ("", ";Jet1 true Pt [GeV]; Jet1 reco Pt [GeV]", 125, 0., 125., 125, 0., 125.), categories=signal_category)
analysis.book_histogram_2D("true_quark2_Pt_vs_R2Jet_sel2_Pt", "true_quark2_Pt", "R2Jet_sel2_Pt", ("", ";Jet2 true Pt [GeV]; Jet2 reco Pt [GeV]", 125, 0., 125., 125, 0., 125.), categories=signal_category)

analysis.book_histogram_1D("true_quark1_Pt", "true_quark1_Pt", ("", ";Jet1 true Pt [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("true_quark1_Px", "true_quark1_Px", ("", ";Jet1 true Px [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("true_quark1_Py", "true_quark1_Py", ("", ";Jet1 true Py [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("true_quark2_Pt", "true_quark2_Pt", ("", ";Jet2 true Pt [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("true_quark2_Px", "true_quark2_Px", ("", ";Jet2 true Px [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("true_quark2_Py", "true_quark2_Py", ("", ";Jet2 true Py [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("R2Jet_sel1_Pt", "R2Jet_sel1_Pt", ("", ";Jet1 reco Pt [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("R2Jet_sel1_Px", "R2Jet_sel1_Px", ("", ";Jet1 reco Px [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("R2Jet_sel1_Py", "R2Jet_sel1_Py", ("", ";Jet1 reco Py [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("R2Jet_sel2_Pt", "R2Jet_sel2_Pt", ("", ";Jet2 reco Pt [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("R2Jet_sel2_Px", "R2Jet_sel2_Px", ("", ";Jet2 reco Px [GeV]", 125, 0., 125.), categories=signal_category)
analysis.book_histogram_1D("R2Jet_sel2_Py", "R2Jet_sel2_Py", ("", ";Jet2 reco Py [GeV]", 125, 0., 125.), categories=signal_category)

In [27]:
analysis.book_histogram_2D("R2Jet_sel1_delta_Px_vs_delta_Py", "R2Jet_sel1_delta_Px", "R2Jet_sel1_delta_Py", ("", ";Jet1 #Delta Px [GeV]; Jet1 #Delta Py [GeV]", 150, -15., 15., 150, -15., 15.), categories=signal_category)
analysis.book_histogram_2D("R2Jet_sel2_delta_Px_vs_delta_Py", "R2Jet_sel2_delta_Px", "R2Jet_sel2_delta_Py", ("", ";Jet2 #Delta Px [GeV]; Jet2 #Delta Py [GeV]", 150, -15., 15., 150, -15., 15.), categories=signal_category)

In [28]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec)

In [29]:
%%time
analysis.run()

CPU times: user 7min 44s, sys: 44.4 s, total: 8min 28s
Wall time: 3min


In [30]:
# if write_outputs:
    # analysis.check_snapshots("events", output_path, checked_output_meta)

In [31]:
h = analysis.draw_summed_unscaled_histograms("true_quark1_Pt_vs_R2Jet_sel1_Pt", signal_category[0])
h = analysis.draw_summed_unscaled_histograms("true_quark2_Pt_vs_R2Jet_sel2_Pt", signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet_sel1_Pt", "true_quark1_Pt"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet_sel2_Pt", "true_quark2_Pt"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet_sel2_Px", "true_quark2_Px"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["R2Jet_sel2_Py", "true_quark2_Py"], signal_category[0])

In [32]:
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_P", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Pt", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Pt2", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Pxy", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_PxPy", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_PxPy2", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Px", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Py", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Pz", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_phi", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Px_vs_delta_Py", signal_category[0])

3.716312095934405
2.5923214072954197
106.85106187680992
81.13212684663067
3.148278046975039
106.85106187680992
2.4415862253917555
2.440516651863814
3.636479910740465
0.03286608447066217


In [33]:
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel1_delta_P", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel1_delta_theta", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel1_delta_phi", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_P", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_theta", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_phi", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("iso_lep_delta_P", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("iso_lep_delta_theta", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("iso_lep_delta_phi", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("nu_delta_P", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("nu_delta_theta", signal_category[0], RMS90=True)
h = analysis.draw_summed_unscaled_histograms("nu_delta_phi", signal_category[0], RMS90=True)

3.5183576538630854
0.04546709380230548
0.03927210348732313
3.716312095934405
0.03540176005642007
0.03286608447066217
0.8514297597873062
3.3343815981046026e-05
8.389812177031012e-05
5.3357175530255985
0.07125662037069991
0.052182515606136064


In [34]:
f = ROOT.TF1("f", ROOT.DoubleSidedCrystalballFunction, -0.15, 0.15, 7)
f.SetParameter(0, h.GetMean() - h.GetRMS())
f.SetParameter(1, h.GetMean() + h.GetRMS())
f.SetParameter(2, 1.5)
f.SetParameter(3, 1.5)
f.SetParameter(4, h.GetMean())
f.SetParameter(5, h.GetRMS()/2)
f.SetParameter(6, h.GetEntries() / h.GetNcells())
f.Print("V")
h.Fit(f, "S")
c = ROOT.TCanvas()
h.Draw("e1")
c.Draw()

Compiled based function: f  based on a functor object.  Ndim = 1, Npar = 7
List of  Parameters: 
                   p0 =   -0.070798 
                   p1 =    0.070149 
                   p2 =    1.500000 
                   p3 =    1.500000 
                   p4 =   -0.000325 
                   p5 =    0.035237 
                   p6 =  7874.809211 
****************************************
         Invalid FitResult  (status = 3 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =        42068
NDf                       =          143
Edm                       =       255222
NCalls                    =          626
p0                        =     0.795891   +/-   1.06006     
p1                        =      1.02705   +/-   1.06007     
p2                        =       4.0866   +/-   1.0604      
p3                        =      13.6143   +/-   1           
p4                        =    0.0037641   +/-   1.06006     
p5             

Warning in <Fit>: Abnormal termination of minimization.


In [35]:
analysis.draw_summed_unscaled_histograms("R2Jet_sel1_delta_Pt_vs_Pt", signal_category[0])
analysis.draw_summed_unscaled_histograms("R2Jet_sel2_delta_Pt_vs_Pt", signal_category[0])

In [36]:
# analysis.draw_summed_unscaled_histograms("true_hadronic_W_postCRC_delta_E", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_hadronic_W_postCRC_delta_P", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_hadronic_W_postCRC_delta_Pt", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_hadronic_W_postCRC_delta_Pz", signal_category[0])

# analysis.draw_summed_unscaled_histograms("true_quark1_postCRC_delta_E", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_quark1_postCRC_delta_P", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_quark1_postCRC_delta_Pt", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_quark1_postCRC_delta_Pz", signal_category[0])

# analysis.draw_summed_unscaled_histograms("true_quark2_postCRC_delta_E", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_quark2_postCRC_delta_P", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_quark2_postCRC_delta_Pt", signal_category[0])
# analysis.draw_summed_unscaled_histograms("true_quark2_postCRC_delta_Pz", signal_category[0])

In [37]:
analysis.draw_summed_unscaled_histograms("iso_lep_M", signal_category[0])
analysis.draw_summed_unscaled_histograms("R2Jet1_M", signal_category[0])
analysis.draw_summed_unscaled_histograms("R2Jet2_M", signal_category[0])

In [38]:
analysis.draw_summed_unscaled_histograms("nu_Pz_vst", signal_category[0])
analysis.draw_summed_unscaled_histograms("gamma_nu_Pz_vst", signal_category[0])

In [39]:
analysis.draw_summed_unscaled_histograms("isr_Pz_rvst", signal_category[0])
analysis.draw_summed_unscaled_histograms("isr_Pz_wrvst", signal_category[0])

In [40]:
analysis.compare_summed_histograms_unscaled(["gamma_PzQ_diff", "wrong_gamma_PzQ_diff"], signal_category[0], logY=True)
analysis.compare_summed_histograms_unscaled(["gamma_Pz_diff", "wrong_gamma_Pz_diff"], signal_category[0], logY=True)
analysis.compare_summed_histograms_unscaled(["gamma_nu_lep_M", "wrong_gamma_nu_lep_M", "true_nu_lep_M"], signal_category[0], logY=True)
analysis.compare_summed_histograms_unscaled(["gamma_Pz", "wrong_gamma_Pz", "true_isr_sum_Pz"], signal_category[0], logY=True)
analysis.compare_summed_histograms_unscaled(["gamma_PzQ", "wrong_gamma_PzQ"], signal_category[0], logY=True)

In [41]:
analysis.compare_summed_histograms_unscaled(["nu_delta_theta", "gamma_nu_delta_theta", "wrong_gamma_nu_delta_theta"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_Pz", "gamma_nu_delta_Pz", "wrong_gamma_nu_delta_Pz"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_E", "gamma_nu_delta_E", "wrong_gamma_nu_delta_E"], signal_category[0])
# obviously does not change as it is independent from z component
# analysis.compare_histograms_unscaled(["nu_delta_phi", "gamma_nu_delta_phi"], categories=signal_category)

In [42]:
analysis.draw_histogram("true_lep_E", categories=signal_category, draw_legend=False)

analysis.draw_summed_unscaled_histograms("gamma_Pz", signal_category[0], logY=True)

analysis.draw_summed_unscaled_histograms("true_isr_sum_M", signal_category[0], logY=True)
analysis.draw_summed_unscaled_histograms("true_isr_sum_Pt", signal_category[0], logY=True)
analysis.draw_summed_unscaled_histograms("true_isr_min_E", signal_category[0], logY=True)

analysis.draw_summed_unscaled_histograms("nu_delta_P_scaled", signal_category[0])
# analysis.draw_summed_unscaled_histograms("clean_nu_delta_P_scaled", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_clean_nu_delta_P_scaled", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_ISR_clean_nu_delta_P_scaled", signal_category[0])

analysis.draw_summed_unscaled_histograms("nu_M", signal_category[0])
# analysis.draw_summed_unscaled_histograms("clean_nu_M", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_clean_nu_M", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_ISR_clean_nu_M", signal_category[0])

analysis.draw_summed_unscaled_histograms("nu_delta_E", signal_category[0])
analysis.draw_summed_unscaled_histograms("gamma_nu_delta_E", signal_category[0])
# analysis.draw_summed_unscaled_histograms("clean_nu_delta_E", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_clean_nu_delta_E", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_ISR_clean_nu_delta_E", signal_category[0])

# analysis.draw_summed_unscaled_histograms("clean_nu_delta_P", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_ISR_clean_nu_delta_P", signal_category[0])
# analysis.draw_summed_unscaled_histograms("clean_nu_delta_Pt", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_ISR_clean_nu_delta_Pt", signal_category[0])
# analysis.draw_summed_unscaled_histograms("clean_nu_delta_Pz", signal_category[0])
# analysis.draw_summed_unscaled_histograms("cheat_ISR_clean_nu_delta_Pz", signal_category[0])

# analysis.draw_unscaled_histograms("clean_nu_delta_E", categories=signal_category)
analysis.draw_summed_unscaled_histograms("nu_delta_theta", signal_category[0])
analysis.draw_summed_unscaled_histograms("gamma_nu_delta_theta", signal_category[0])
analysis.draw_summed_unscaled_histograms("nu_delta_phi", signal_category[0])
analysis.draw_summed_unscaled_histograms("gamma_nu_delta_phi", signal_category[0])

In [43]:

analysis.compare_summed_histograms_unscaled(["iso_lep_delta_E", "iso_lep_gamma_delta_E", "iso_lep_brems_delta_E"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_P", "iso_lep_gamma_delta_P", "iso_lep_brems_delta_P"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_theta", "iso_lep_gamma_delta_theta", "iso_lep_brems_delta_theta"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["iso_lep_delta_phi", "iso_lep_gamma_delta_phi", "iso_lep_brems_delta_phi"], signal_category[0])

# analysis.compare_histograms_unscaled(["iso_lep_delta_phi_q", "iso_lep_gamma_delta_phi_q", "iso_lep_brems_delta_phi_q"], categories=signal_category)


In [44]:
# analysis.compare_histograms_unscaled(["true_lep_E", "iso_lep_E"], categories=signal_category)

In [45]:
for i in [1, 2]:
    for var in ["E", "P", "Pt", "Pz", "theta", "phi"]:
        analysis.compare_summed_histograms_unscaled([f"R2Jet_sel{i}_delta_{var}", f"clean_R2Jet_sel{i}_delta_{var}", f"cheat_clean_R2Jet_sel{i}_delta_{var}"], signal_category[0])

In [46]:
analysis.compare_summed_histograms_unscaled(["nu_delta_E", "clean_nu_delta_E", "cheat_clean_nu_delta_E"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_P", "clean_nu_delta_P", "cheat_clean_nu_delta_P"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_M", "clean_nu_M", "cheat_clean_nu_M"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_theta", "clean_nu_delta_theta", "cheat_clean_nu_delta_theta"], signal_category[0])
analysis.compare_summed_histograms_unscaled(["nu_delta_phi", "clean_nu_delta_phi", "cheat_clean_nu_delta_phi"], signal_category[0])